In [ ]:
import os
import numpy as np
import re
import sys

sys.path.append(os.path.abspath(".."))
import vector_borne_functions as vbf

T = vbf.T
f = vbf.f
g = vbf.g
F = vbf.F



def sum_alpha_i(MGDD_R):
    

    c_1 = 0.012
    c_2 = 975
    
    sum_alpha = MGDD_R + 1/c_1*np.log( (1 + np.exp(-c_1 * (MGDD_R-c_2) ) ) /(1 + np.exp(c_1 * c_2) ) ) 

    return sum_alpha



def get_climatic_arrays(folder_name,day_fraction):

    dimension, a_min, a_max,b_min,b_max = [int(x) for x in re.findall(r"[-+]?\d*\.?\d+", folder_name)][:5]

    # a and b values arrays
    a_values = np.linspace(a_min,a_max,dimension)
    b_values = np.linspace(b_min,b_max,dimension)

    # factor to work in years 
    factor = 365

    mean_f_t_array = np.zeros((dimension,dimension))
    mean_g_t_array = np.zeros((dimension,dimension))

    times = times = np.array(range(365*day_fraction))/(365*day_fraction)

    for row in range(dimension):
        #print(row)
            for column in range(dimension):
                            
                a,b = a_values[row], b_values[column]

                if b>12 and b>a:

                    T_t = T(times,a,b)
                    
                    f_t = f(T_t)*365
                    g_t = g(T_t)*365

                    """
                    f_t = daily_mean_arrays(f_t,day_fraction)*factor
                    g_t = daily_mean_arrays(g_t,day_fraction)*factor
                    """
                
                    f_t_mean = np.mean(f_t[:day_fraction*365])
                    g_t_mean = np.mean(g_t[:day_fraction*365])
                        
                    mean_f_t_array[row,column] = f_t_mean      
                    mean_g_t_array[row,column] = g_t_mean
                
                if b<a or b<12:

                    mean_f_t_array[row,column] = -0.1
                    mean_g_t_array[row,column] = -0.1

    return mean_f_t_array, mean_g_t_array



def sum_g(n,mean_g_t_array, mean_f_t_array):

    MGDD_R = 1500.
    Delta_MGDD=MGDD_R/n
    MGDD = np.array([x*Delta_MGDD for x in range(n+1)])

    s=0
    for i in range(1,n-1):
        s_=0
        for j in range(i-1,n-2):
        
            s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))

        s+=F(MGDD[i])*s_

    return s*Delta_MGDD
    



def get_r_0(f_m_array, g_m_array,sum_g_f, Alpha, beta, mu, Gamma):

    MGDD_R = 1500.
    alpha_sum = sum_alpha_i(MGDD_R)

    R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum + sum_g_f ) / f_m_array )  * (1-(g_m_array / f_m_array)) #/(1-(mean_g_t_array / mean_f_t_array)**(n-1))
    R_0_aproximation_array [R_0_aproximation_array <0] = 0


    mask = (f_m_array<0) & (g_m_array<0) 
    R_0_aproximation_array[mask] = -0.1

    return R_0_aproximation_array





#list to store the failed jobs (simulation data that should be in R_folders and do not)
failed_jobs = []


# path to the directory with the data of R_{\infty}}
cwd = os.getcwd()

parent_dir = cwd + '/R_infinity_arrays' #+ '/R_0_arrays'


# list of folder in the folder "R_0_arrays"
R_folders = os.listdir(parent_dir)
print(R_folders)
#chosing one folder
#folder_name = input(f'las carpetas que hay son {R_folders} elige una:')
folder_name = R_folders[0]

# path to the folder with data of the simulations with "dimension", "a_min", ...
directory = parent_dir + '/' + folder_name

# names of the arrays in the folder with data of the simulations with "dimension", "a_min", ...
# these arrays have names Arr_{Alpha}_{beta}_{mu}_{Gamma}

folder_arrays = os.listdir(directory)

# parameters of the simulation that give rise to the data in each array
disordered_parameters = [[float(x) for x in re.findall(r"[-+]?\d*\.?\d+", y)] for y in folder_arrays if re.search(r'\d', y)]


disordered_parameters = np.array(disordered_parameters)

ordered_parameters = np.loadtxt(directory + '/params.txt')

parameters = np.array([row for row in ordered_parameters  if  any((row == disordered_parameters).all(axis=1))])

# list to store the r_infty arrays 
R_inf_arrays = []
# list to store the r_0 arrays
R_0_arrays = []


parameters = parameters[(parameters[:,0]*parameters[:,1]/(parameters[:,2]*parameters[:,3])<20)]



day_fraction = re.search(r"dfraction_(\d+)", folder_name)
n = re.search(r"n(\d+)", folder_name)

day_fraction = int(day_fraction.group(1)) if day_fraction else None
n = int(n.group(1)) if n else None


f_m_array, g_m_array = get_climatic_arrays(folder_name,day_fraction)

sum_g_f = sum_g(n,g_m_array, f_m_array)

for parameter_combination in parameters:

    Alpha, beta, mu, Gamma, N_H, N_v = parameter_combination
    R_inf_arrays.append(np.loadtxt(directory + f'/R_inf_values_{Alpha}_{beta}_{mu}_{Gamma}_{N_H}_{N_v}.txt'))

    r_0 = get_r_0(f_m_array, g_m_array, sum_g_f, Alpha, beta, mu, Gamma)
    R_0_arrays.append(r_0)


flat_r_infty = np.concatenate(R_inf_arrays).ravel()
flat_r_0 = np.concatenate(R_0_arrays).ravel()

flat_r_infty[flat_r_infty<0] = np.nan
flat_r_0[flat_r_0<0] = np.nan


np.savetxt('flat_r_infty.txt',flat_r_infty)
np.savetxt('flat_r_0.txt',flat_r_0)


#     figure 2
def get_r_0_a_b(a, b, Alpha, beta, Gamma, mu):

    times = np.array(range(365))/365

    T_t = T(times,a,b)
                    
    f_t = f(T_t)*365
    g_t = g(T_t)*365


    f_m_array = np.mean(f_t)
    g_m_array = np.mean(g_t)


    sum_g_f = sum_g(n,g_m_array, f_m_array)



    MGDD_R = 1500.
    alpha_sum = sum_alpha_i(MGDD_R)

    R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum + sum_g_f ) / f_m_array )  * (1-(g_m_array / f_m_array)) /(1-(g_m_array / f_m_array)**(n-1))
    
    return R_0_aproximation_array



# list of parameters: (T_min,T_max,Alpha,beta,Gamma,mu)
a_b_s = (-6, 25 , 4.015, 2., 0.2, 8.03), (3, 23.47 , 4.015, 2., 0.2, 8.03), (7, 14, 4.015, 0.65, 0.5, 6.), (8, 19, 4.015, 0.65, 0.5, 6.), (25, 31, 4.015, 0.65, 0.5, 6.)

R_0_a_b_s = []

for parameter_combination in a_b_s:

    a,b,Alpha,beta,Gamma,mu = parameter_combination

    R_0_a_b = get_r_0_a_b(a, b, Alpha, beta, Gamma, mu)

    R_0_a_b_s.append(R_0_a_b)

R_0_a_b_s = np.array(R_0_a_b_s)

R_0_a_b_s = R_0_a_b_s.round(2)


np.savetxt('R_0_a_b_s.txt',R_0_a_b_s)



['dimension_70_arage_-20_26_brange_12_50_dfraction_10_n200_Iv_']


/tmp/ipykernel_3113629/2067850733.py:86: RuntimeWarning: invalid value encountered in divide
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_3113629/2067850733.py:86: RuntimeWarning: overflow encountered in power
  s_+= (mean_g_t_array/mean_f_t_array)**(n-(2+j))
/tmp/ipykernel_3113629/2067850733.py:100: RuntimeWarning: invalid value encountered in divide
  R_0_aproximation_array = Alpha * beta / (mu * Gamma) * ( 1 + Gamma *( alpha_sum + sum_g_f ) / f_m_array )  * (1-(g_m_array / f_m_array)) #/(1-(mean_g_t_array / mean_f_t_array)**(n-1))
